In [1]:
import json
import os
import urllib

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, 'w') as f:
            f.write(text_data)
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

file_path = "datasets/instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)
data = download_and_load_file(file_path, url)
print(len(data))
            

1100


In [2]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task."
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry['input'] else ""
    )
    return instruction_text + input_text

model_input = format_input(data[999])
desired_output = data[999]['output']
print(model_input + desired_output)

Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?An antonym of 'complicated' is 'simple'.


In [3]:
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.10)
valid_portion = len(data) - train_portion - test_portion

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
valid_data = data[train_portion + test_portion:]
print(f"Train data size: {len(train_data)}")
print(f"Test data size: {len(test_data)}")
print(f"Validation data size: {len(valid_data)}")

Train data size: 935
Test data size: 110
Validation data size: 55


In [4]:
import torch
from torch.utils.data import Dataset
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        super().__init__()
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.encoded_texts[idx]

## 自定义批处理聚合函数

In [5]:
def custom_collate_fn(
        batch,
        pad_token_id=50256,
        ignore_index=-100,
        allowed_max_length=None,
        device="cpu"
):
    batch_max_length = max(len(item) + 1 for item in batch)
    inputs_lst = []
    target_lst = []
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = (
            new_item + [pad_token_id] * (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index
        if allowed_max_length is not None:
            targets = targets[:allowed_max_length]
            inputs = inputs[:allowed_max_length]
        inputs_lst.append(inputs)
        target_lst.append(targets)
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(target_lst).to(device)
    return inputs_tensor, targets_tensor

In [6]:
input1 = [0, 1, 2, 3, 4]
input2 = [5, 6, 7]
input3 = [9, 10]

batch = (
    input1,
    input2,
    input3
)
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6,     7, 50256, 50256],
        [    9,    10, 50256, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6,     7, 50256,  -100,  -100],
        [   10, 50256,  -100,  -100,  -100]])


## 探索为什么-100

In [7]:
logit1 = torch.tensor([[-1.0, 1.0],
                       [-0.5, 1.5],
                       [-0.5, 1.5]])
target1 = torch.tensor([0, 1, -100])
loss1 = torch.nn.functional.cross_entropy(logit1, target1)
print(loss1)

tensor(1.1269)


## 创建数据loader

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from functools import partial
customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)


In [9]:
from torch.utils.data import DataLoader
import tiktoken
num_workers = 0
batch_size = 8
tokenizer = tiktoken.get_encoding("gpt2")
torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer=tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    collate_fn=customized_collate_fn,
    drop_last=True
)
valid_dataset = InstructionDataset(valid_data, tokenizer=tokenizer)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    collate_fn=customized_collate_fn,
    drop_last=True
)
test_dataset = InstructionDataset(test_data, tokenizer=tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    collate_fn=customized_collate_fn,
    drop_last=True
)


In [10]:
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

torch.Size([8, 61]) torch.Size([8, 61])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 73]) torch.Size([8, 73])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 72]) torch.Size([8, 72])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 77]) torch.Size([8, 77])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 66]) torch.Size([8, 66])
torch.Size([8, 83]) torch.Size([8, 83])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 68]) torch.Size([8, 68])


## 加载预训练模型

In [23]:
from gpt_download import download_and_load_gpt2
from models.GPTModel import GPTModel
from utils import load_weights_into_gpt, generate, text_to_token_ids, token_ids_to_text

BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "dropout": 0.1,
    "bias": True,
}
model_config = {
    "gpt2-small" : {"emb_dim":768, "num_layers":12, "num_heads":12},
    "gpt2-medium" : {"emb_dim":1024, "num_layers":24, "num_heads":16},
    "gpt2-large" : {"emb_dim":1280, "num_layers":36, "num_heads":20},
    "gpt2-xl" : {"emb_dim":1600, "num_layers":48, "num_heads":25}
}

BASE_CONFIG.update(model_config["gpt2-medium"])
settings, params = download_and_load_gpt2(
    model_size="355M",
    models_dir="../models/"
)
model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

File already exists and is up-to-date: ../models/355M/checkpoint
File already exists and is up-to-date: ../models/355M/encoder.json
File already exists and is up-to-date: ../models/355M/hparams.json
File already exists and is up-to-date: ../models/355M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: ../models/355M/model.ckpt.index
File already exists and is up-to-date: ../models/355M/model.ckpt.meta
File already exists and is up-to-date: ../models/355M/vocab.bpe


2026-01-24 16:41:38.576602: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 205852672 exceeds 10% of free system memory.


GPTModel(
  (tok_embedding): Embedding(50257, 1024)
  (pos_embedding): Embedding(1024, 1024)
  (dropout): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (attention): MultiHeadAttention(
        (w_q): Linear(in_features=1024, out_features=1024, bias=True)
        (w_k): Linear(in_features=1024, out_features=1024, bias=True)
        (w_v): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ffn): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attention): MultiHeadAttention(
        (w_q): Line

In [12]:
torch.manual_seed(123)
input_text = format_input(valid_data[0])
print(input_text)

Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'


In [13]:
token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256,
)
generated_text = token_ids_to_text(token_ids, tokenizer)
response_test = generated_text[len(input_text):].strip()
print("Generated response:\n", response_test)

Generated response:
 Convert the passive to active sentence: 'I was helping my mother open her oven at the end of the month to hot the insides.'

Create an array of function


## 微调大模型

In [21]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)[:, -1, :]
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0
    if len(data_loader) == 0:
        return float('nan')
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, eval_iter)
        valid_loss = calc_loss_loader(val_loader, model, device, eval_iter)
    model.train()
    return train_loss, valid_loss

def generate_and_print_sample(model, tokenizer, start_context, device):
    model.eval()
    context_size = model.pos_embedding.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text(model, encoded, 50, context_size)
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace('\n', ' ')) 
    model.train()

def generate_text(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

def calc_loss(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1),
        target_batch.flatten()
    )
    return loss

def train_model_simple(model, train_loader, val_loader, optimizer, device, 
                       epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, step = 0, -1
    for epoch in range(epochs):
        model.train()
        for inputs, targets in train_loader:
            optimizer.zero_grad()
            loss = calc_loss(inputs, targets, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen += inputs.numel()
            step += 1

            if step % eval_freq == 0:
                model.eval()
                with torch.no_grad():
                    train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                    train_losses.append(train_loss)
                    val_losses.append(val_loss)
                    track_tokens_seen.append(tokens_seen)
                    print(f"Epoch: {epoch}, Step: {step}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        generate_and_print_sample(model, tokenizer, start_context, device)
    return train_losses, val_losses, track_tokens_seen


In [24]:
model.to(device)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    valid_loss = calc_loss_loader(valid_loader, model, device, num_batches=5)
print(f"Initial Train Loss: {train_loss:.4f}, Initial Valid Loss: {valid_loss:.4f}")

Initial Train Loss: 4.1410, Initial Valid Loss: 4.0418


In [26]:
import time
start_time = time.time()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-1)
num_epochs = 2

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, valid_loader, optimizer, device,
    epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(valid_data[0]),
    tokenizer=tokenizer
)
end_time = time.time()
print(f"Training time: {end_time - start_time:.2f} seconds")

Epoch: 0, Step: 0, Train Loss: 1.0269, Val Loss: 1.0612
Epoch: 0, Step: 5, Train Loss: 0.8649, Val Loss: 0.9714
Epoch: 0, Step: 10, Train Loss: 0.8299, Val Loss: 0.9232
Epoch: 0, Step: 15, Train Loss: 0.8321, Val Loss: 0.8865
Epoch: 0, Step: 20, Train Loss: 0.7900, Val Loss: 0.8515
Epoch: 0, Step: 25, Train Loss: 0.8260, Val Loss: 0.8377
Epoch: 0, Step: 30, Train Loss: 0.6458, Val Loss: 0.8220
Epoch: 0, Step: 35, Train Loss: 0.7116, Val Loss: 0.8066
Epoch: 0, Step: 40, Train Loss: 0.7541, Val Loss: 0.8058
Epoch: 0, Step: 45, Train Loss: 0.6877, Val Loss: 0.7926
Epoch: 0, Step: 50, Train Loss: 0.6462, Val Loss: 0.7676
Epoch: 0, Step: 55, Train Loss: 0.6255, Val Loss: 0.7512
Epoch: 0, Step: 60, Train Loss: 0.5856, Val Loss: 0.7343
Epoch: 0, Step: 65, Train Loss: 0.5521, Val Loss: 0.7411
Epoch: 0, Step: 70, Train Loss: 0.6850, Val Loss: 0.7274
Epoch: 0, Step: 75, Train Loss: 0.5572, Val Loss: 0.7227
Epoch: 0, Step: 80, Train Loss: 0.5516, Val Loss: 0.7154
Epoch: 0, Step: 85, Train Loss: 0

## 测试并保留测试数据集上的答复

In [41]:
for entry in test_data[:3]:
    input_text = format_input(entry)
    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256,
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_test = generated_text[len(input_text):].replace("### Response:", "").strip()
    print("输入内容:\n", input_text)
    print("生成答复:\n", response_test)
    print("预期答复:\n", entry['output'])
    print("-" * 80)

输入内容:
 Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
Rewrite the sentence using a simile.

### Input:
The car is very fast.
生成答复:
 The car was so fast it could make you dizzy.
预期答复:
 The car is as fast as lightning.
--------------------------------------------------------------------------------
输入内容:
 Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
What type of cloud is typically associated with thunderstorms?
生成答复:
 A typical thunderstorm typically features a strong, persistent thunderstorm with high winds. An ionized cloud typically precipitates in the form of a hydrogen bond.
预期答复:
 The type of cloud typically associated with thunderstorms is cumulonimbus.
--------------------------------------------------------------------------------
输入内容:
 Below is an instruction that describes a task.Write a response that appropriately completes

In [42]:
from tqdm import tqdm
for i, entry in tqdm(enumerate(test_data), total=len(test_data)):
    input_text = format_input(entry)
    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256,
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_test = generated_text[len(input_text):].replace("### Response:", "").strip()
    test_data[i]['generated_response'] = response_test
with open("datasets/instruction-data-with-responses.json", "w") as f:
    json.dump(test_data, f, indent=4)

100%|██████████| 110/110 [00:38<00:00,  2.82it/s]


In [43]:
print(test_data[0])

{'instruction': 'Rewrite the sentence using a simile.', 'input': 'The car is very fast.', 'output': 'The car is as fast as lightning.', 'generated_response': 'The car is fast as a bullet.'}
